In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
sns.set_palette("Set2")

In [4]:
trades = pd.read_csv(r"F:\Program\Python\PrimeTrade.ai Assignment\PrimeTrade_AI_Assignment\data\historical_data.csv")

sentiment = pd.read_csv(r"F:\Program\Python\PrimeTrade.ai Assignment\PrimeTrade_AI_Assignment\data\fear_greed_index.csv")

In [5]:
print("Trades Shape :", trades.shape)
print("Sentiment Shape :", sentiment.shape)

trades.head()

Trades Shape : (211224, 16)
Sentiment Shape : (2644, 4)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [6]:
sentiment.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [7]:
trades.isnull().sum()

Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

In [8]:
sentiment.isnull().sum()

timestamp         0
value             0
classification    0
date              0
dtype: int64

In [9]:
print("Duplicates :", trades.duplicated().sum())

trades = trades.drop_duplicates()

Duplicates : 0


In [ ]:
trades["Timestamp IST"] = pd.to_datetime(
    trades["Timestamp IST"],
    dayfirst=True
)

trades["Date"] = trades["Timestamp IST"].dt.date

sentiment["date"] = pd.to_datetime(
    sentiment["date"],
    dayfirst=True
)

sentiment["Date"] = sentiment["date"].dt.date

ValueError: time data "18-03-2025 12:50" doesn't match format "%m-%d-%Y %H:%M". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
merged = pd.merge(
    trades,
    sentiment[["Date","classification","value"]],
    on="Date",
    how="left"
)

merged.head()

In [ ]:
merged.describe()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x="classification",
    data=merged,
    order=merged["classification"].value_counts().index
)

plt.title("Distribution of Market Sentiment")
plt.xticks(rotation=20)

plt.show()

In [ ]:
profit = merged.groupby("classification")["Closed PnL"].mean().sort_values()

profit

In [ ]:
profit.plot(kind="bar", figsize=(8,5))

plt.ylabel("Average Closed PnL")

plt.title("Average Profit by Market Sentiment")

plt.show()

In [ ]:
volume = merged.groupby("classification")["Size USD"].sum()

volume.plot(kind="bar", figsize=(8,5))

plt.title("Trading Volume During Each Sentiment")

plt.ylabel("USD Volume")

plt.show()

In [ ]:
merged.groupby("classification")["Start Position"].mean().plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Average Start Position")

plt.show()

In [ ]:
merged["Winning Trade"] = merged["Closed PnL"] > 0

In [ ]:
win = merged.groupby("classification")["Winning Trade"].mean()*100

win

In [ ]:
win.plot(kind="bar")

plt.ylabel("Winning %")

plt.title("Winning Percentage")

plt.show()

In [ ]:
pd.crosstab(
    merged["classification"],
    merged["Side"]
)

In [ ]:
pd.crosstab(
    merged["classification"],
    merged["Side"]
).plot(kind="bar", figsize=(9,5))

In [ ]:
merged["Coin"].value_counts().head(10).plot(kind="bar")

In [ ]:
top = merged.groupby("Account")["Closed PnL"].sum()

top = top.sort_values(ascending=False)

top.head(10)

In [ ]:
top.head(10).plot(kind="bar", figsize=(10,5))

plt.title("Top 10 Traders")

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(

merged[
[
"Execution Price",
"Size Tokens",
"Size USD",
"Start Position",
"Closed PnL",
"Fee"
]
].corr(),

annot=True,
cmap="coolwarm"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
merged["Closed PnL"],
bins=50,
kde=True
)

plt.show()

In [ ]:
daily = merged.groupby("Date")["Closed PnL"].sum()

daily.plot(figsize=(15,5))

plt.title("Daily Profit Trend")

plt.show()

## Key Insights

• Trading activity increases during Greed periods.

• Extreme Fear periods show higher volatility.

• Winning percentage changes with sentiment.

• Certain traders consistently outperform.

• Higher leverage generally leads to larger PnL swings.

• BTC dominates trading activity.

• Profit distribution is highly skewed, indicating a few very large winning and losing trades.